**Important:** These notebooks are sequential sections of the master notebook. Run them in order in the **same Colab runtime**; restarting the runtime clears required variables. The self-contained master notebook is recommended for a fresh run. The final-test section requires the locked-predictions GitHub commit before unlocking test labels.

# Yapay Sinir Ağları — MLP Midterm | Google Colab Master Notebook
**Student last five digits:** 12902 · **Base seed:** 14928 · **Dataset:** UCI `bank-additional.csv` (4,119 rows).

**Important:** This notebook is an executable research workflow, **not precomputed results**. It can require a long CPU runtime (144+ MLP fits, including final selection and stability checks). Google Colab sessions can disconnect; keep Drive backups. The independent test is deliberately locked until a separate GitHub commit. Run cells sequentially; do not use `Runtime → Run all` across the lock checkpoint.

Outputs are created under `/content/MLP_Midterm_14928/`. At the end, download the project ZIP. All numerical results must come from actual execution.


In [ ]:
import os, json, random, time, warnings, pathlib, platform
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (average_precision_score, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, confusion_matrix, roc_curve, precision_recall_curve)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
import torch
from torch import nn
ROOT=pathlib.Path('/content/MLP_Midterm_14928') if pathlib.Path('/content').exists() else pathlib.Path.cwd()
ROOT.mkdir(parents=True,exist_ok=True)
for folder in ['results','figures','configs']: (ROOT/folder).mkdir(exist_ok=True)
STUDENT_LAST5=12902  # Change if needed
SEED=STUDENT_LAST5+2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Seed:',SEED,'torch:',torch.__version__,'sklearn:',__import__('sklearn').__version__)

import sklearn, sys
with open(ROOT/'results/environment.json','w') as f: json.dump({'python':sys.version,'torch':torch.__version__,'sklearn':sklearn.__version__,'numpy':np.__version__,'platform':platform.platform(),'seed':SEED,'device':'cpu'},f,indent=2)
torch.set_num_threads(2)


In [ ]:
from urllib.request import urlretrieve
from zipfile import ZipFile
DATA=ROOT/'bank-additional.csv'
if not DATA.exists():
    archive=ROOT/'bank-additional.zip'
    urlretrieve('https://archive.ics.uci.edu/static/public/222/bank+marketing.zip',archive)
    with ZipFile(archive) as z:
        nested=z.read('bank-additional.zip')
    import io
    with ZipFile(io.BytesIO(nested)) as z:
        DATA.write_bytes(z.read('bank-additional/bank-additional.csv'))
df=pd.read_csv(DATA,sep=';')
assert df.shape==(4119,21),df.shape
assert set(df.y.unique())=={'yes','no'}
y=df.y.map({'no':0,'yes':1}).astype(int)
X=df.drop(columns=['y','duration'])
idx=np.arange(len(df))
dev_idx,test_idx=train_test_split(idx,test_size=.2,stratify=y,random_state=SEED)
Xdev,ydev=X.iloc[dev_idx].reset_index(drop=True),y.iloc[dev_idx].reset_index(drop=True)
Xtest=X.iloc[test_idx].reset_index(drop=True)  # DO NOT ACCESS TEST LABELS BEFORE LOCKED COMMIT
split=pd.DataFrame({'original_index':np.r_[dev_idx,test_idx],'split':['development']*len(dev_idx)+['test']*len(test_idx),'label':np.r_[y.iloc[dev_idx],np.repeat(np.nan,len(test_idx))]})
split.to_csv(ROOT/'results/dataset_split.csv',index=False)
print('Development',len(dev_idx),'Test',len(test_idx))

assert 'duration' not in Xdev.columns and 'duration' not in Xtest.columns
print('Test labels intentionally unavailable until commit gate.')


## 1. Development-only EDA

In [ ]:
print('Duplicates (development):',Xdev.duplicated().sum())
print('Class distribution:');display(ydev.value_counts().rename(index={0:'no',1:'yes'}).to_frame('count'))
print('Unknown categories:');display((Xdev=='unknown').sum().sort_values(ascending=False).to_frame('unknown_count'))
num=Xdev.select_dtypes(include=np.number).columns.tolist(); cat=Xdev.select_dtypes(exclude=np.number).columns.tolist()
display(Xdev[num].describe().T)
display(Xdev[cat].nunique().to_frame('unique_values'))
fig,ax=plt.subplots();ydev.value_counts().sort_index().plot.bar(ax=ax);ax.set_title('Development class distribution');fig.savefig(ROOT/'figures/class_distribution.png',bbox_inches='tight');plt.show()
for c in num:
 fig,axs=plt.subplots(1,2,figsize=(9,3));Xdev[c].hist(ax=axs[0],bins=30);Xdev.boxplot(column=c,ax=axs[1]);fig.suptitle(c);fig.tight_layout();fig.savefig(ROOT/f'figures/eda_{c}.png');plt.close(fig)
fig,ax=plt.subplots(figsize=(8,6));corr=Xdev[num].corr();im=ax.imshow(corr,vmin=-1,vmax=1,cmap='coolwarm');ax.set_xticks(range(len(num)),num,rotation=90);ax.set_yticks(range(len(num)),num);fig.colorbar(im);fig.tight_layout();fig.savefig(ROOT/'figures/correlation.png');plt.show()
for c in cat: Xdev[c].value_counts().head(12).to_csv(ROOT/f'results/frequency_{c}.csv')

# Additional reproducible EDA tables and figures (development only)
summary=Xdev[num].describe().T
summary['IQR']=summary['75%']-summary['25%']
summary.to_csv(ROOT/'results/numeric_descriptive_statistics.csv')
(Xdev.isna().sum()).to_csv(ROOT/'results/missing_counts.csv')
(Xdev=='unknown').sum().to_csv(ROOT/'results/unknown_counts.csv')
q1=Xdev[num].quantile(.25);q3=Xdev[num].quantile(.75);iqr=q3-q1
outlier=((Xdev[num].lt(q1-1.5*iqr)) | (Xdev[num].gt(q3+1.5*iqr))).sum()
outlier.to_csv(ROOT/'results/iqr_outlier_flags.csv')
print('IQR outlier flags (descriptive only, no rows deleted):');display(outlier.to_frame('flags'))
for c in cat[:5]:
 fig,ax=plt.subplots(figsize=(8,3));Xdev[c].value_counts().head(10).plot.bar(ax=ax);ax.set_title('Development: '+c);fig.tight_layout();fig.savefig(ROOT/f'figures/category_{c}.png');plt.close(fig)
for c in ['age','campaign','euribor3m']:
 fig,ax=plt.subplots(figsize=(7,3));pd.DataFrame({'feature':Xdev[c],'target':ydev}).boxplot(column='feature',by='target',ax=ax);fig.suptitle('');fig.tight_layout();fig.savefig(ROOT/f'figures/by_target_{c}.png');plt.close(fig)
print('Saved histograms, boxplots, category charts and statistics in figures/ and results/.')


### Pre-registered hypotheses (before outer CV)
**H1:** StandardScaler may improve AP compared with MinMaxScaler.  
**H2:** AdamW, Adam and SGD+Momentum have different convergence behavior at the same learning rate.  
**H3:** Two hidden layers may improve AP but may increase overfitting.  
`unknown` is a distinct category; `pdays=999` means no previous contact and is preserved. MinMax normalization and z-score standardization are different operations. These hypotheses must be evaluated, not presumed true.


In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(10,2));ax.axis('off')
labels=['4119 rows\nremove duration','80% Development\n20% hidden test','3× outer CV\n3× inner CV','12 candidates\nAP selection','Final 3 seeds\nlock predictions','Commit → Test\nBootstrap CI']
for i,s in enumerate(labels):
 x=i/6+.008
 ax.text(x,.5,s,ha='left',va='center',fontsize=8,bbox={'boxstyle':'round','fc':'#edf2f7','ec':'#64748b'})
fig.savefig(ROOT/'figures/nested_cv_flow.png',bbox_inches='tight');plt.show()
